Part 2:
Can be run from top to bottom

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np
from sklearn.compose import ColumnTransformer
import joblib, os


In [ ]:
df_train = pd.read_csv("processed_data/fakenews_995k_train.csv", nrows= 80000)
df_test = pd.read_csv("processed_data/fakenews_995k_test.csv", nrows= 10000)
df_val = pd.read_csv("processed_data/fakenews_995k_val.csv", nrows= 10000)

C:\Users\Julius supercomputer\AppData\Local\Temp\ipykernel_21780\47150615.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("processed_data/fakenews_995k_train.csv", nrows= 80000)


In [ ]:
#I tried loading in chunks but this is incompatible with a countvectorizer and linear regression so i only use the columns that will be necessary for the tests
df_train = pd.read_csv("processed_data/fakenews_995k_train.csv", usecols=['content_tokens_stem','title','meta_keywords','meta_description','type'])
df_test = pd.read_csv("processed_data/fakenews_995k_test.csv", usecols=['content_tokens_stem','title','meta_keywords','meta_description','type'])
df_val = pd.read_csv("processed_data/fakenews_995k_val.csv", usecols=['content_tokens_stem','title','meta_keywords','meta_description','type'])

In [2]:
#data set with "reliable:1" "all others:0"
definitive_labels = [
    "reliable", "fake", "unreliable", "conspiracy",
    "rumor", "clickbait", "junksci"
]
# "political","bias","satire","hate","unknown", NaN removed labels given ambiguity

label_col = ['type']
X_train = df_train[df_train["type"].isin(definitive_labels)]
labels_train = (X_train[label_col] == 'reliable').astype(int)
data_train = X_train["content_tokens_stem"]
X_val = df_val[df_val["type"].isin(definitive_labels)]
data_val = X_val["content_tokens_stem"]
labels_val = (X_val[label_col] == 'reliable').astype(int)
y_train = np.ravel(labels_train)
y_val = np.ravel(labels_val)


In [ ]:
pipe = Pipeline([
    ("vect", CountVectorizer()),
    ("clf", LogisticRegression(random_state=1))
])

param_grid = {
    "vect__max_features": [10000],
    "clf__C": [0.1, 1, 10],
    "clf__max_iter": [1000],
    "clf__solver": ["liblinear", "saga"]
}



grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=1,
    pre_dispatch=1,
    verbose=2
)



In [5]:
grid.fit(data_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation F1:", grid.best_score_)

y_pred = grid.predict(data_val)
print("Validation F1:", f1_score(y_val, y_pred))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END clf__C=0.1, clf__max_iter=1000, clf__solver=liblinear, vect__max_features=10000; total time= 8.8min
[CV] END clf__C=0.1, clf__max_iter=1000, clf__solver=liblinear, vect__max_features=10000; total time= 8.4min
[CV] END clf__C=0.1, clf__max_iter=1000, clf__solver=liblinear, vect__max_features=10000; total time=10.5min


c:\Users\Julius supercomputer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=0.1, clf__max_iter=1000, clf__solver=saga, vect__max_features=10000; total time=12.0min


c:\Users\Julius supercomputer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=0.1, clf__max_iter=1000, clf__solver=saga, vect__max_features=10000; total time=12.0min


...

[CV] END clf__C=10, clf__max_iter=1000, clf__solver=saga, vect__max_features=10000; total time=14.5min


c:\Users\Julius supercomputer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=10, clf__max_iter=1000, clf__solver=saga, vect__max_features=10000; total time=14.3min


c:\Users\Julius supercomputer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=10, clf__max_iter=1000, clf__solver=saga, vect__max_features=10000; total time=14.8min
Best parameters: {'clf__C': 0.1, 'clf__max_iter': 1000, 'clf__solver': 'liblinear', 'vect__max_features': 10000}
Best cross-validation F1: 0.9193918712393195
Validation F1: 0.9230557719627881


In [ ]:
# Save the best model to disk for later use


pipe = Pipeline([
    ("vect", CountVectorizer(max_features=10000)),
    ("clf", LogisticRegression(C=0.1, max_iter=1000, solver="liblinear", random_state=1))
])

pipe.fit(data_train, y_train)

os.makedirs("models", exist_ok=True)
joblib.dump(pipe, "models/logistic_regression_pipeline.pkl")
print("Saved: models/logistic_regression_pipeline.pkl")

In [6]:
X_test = df_test['content_tokens_stem']
y_test = (df_test[label_col] == 'reliable').astype(int)
y_pred = grid.predict(X_test)
print(f1_score(y_test, y_pred=y_pred))


0.8095228322216521


In [ ]:

#best after runthrough of a gridsearch saved not to waste time
Log_model = LogisticRegression(C= 0.1, max_iter= 1000, solver= 'liblinear', random_state= 1) 
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("title", CountVectorizer(max_features=10000), "title"),
        ("meta_keywords", CountVectorizer(max_features=10000), "meta_keywords"),
        ("meta_description", CountVectorizer(max_features=10000), "meta_description")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])



In [9]:
feature_cols_expanded = ['content_tokens_stem','title','meta_keywords','meta_description']

data_train_exp = X_train[feature_cols_expanded].copy()
data_val_exp = X_val[feature_cols_expanded].copy()
data_test_exp = df_test[feature_cols_expanded].copy()
for col in ["content_tokens_stem", "title", "meta_keywords", "meta_description"]:
    data_train_exp[col] = data_train_exp[col].fillna("")
    data_val_exp[col] = data_val_exp[col].fillna("")
    data_test_exp[col] = data_test_exp[col].fillna("")
    
train_Y = np.ravel(labels_train)
val_Y = np.ravel(labels_val)
pipe.fit(data_train_exp, train_Y)
y_pred = pipe.predict(data_val_exp)
print(f1_score(labels_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.9604941623676351
0.8433079654997463


In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("title", CountVectorizer(max_features=10000), "title"),
        ("meta_description", CountVectorizer(max_features=10000), "meta_description")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])
pipe.fit(data_train_exp, y_train)
y_pred = pipe.predict(data_val_exp)
print(f1_score(y_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.9524455293373113
0.8196467245524449


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("meta_keywords", CountVectorizer(max_features=10000), "meta_keywords"),
        ("meta_description", CountVectorizer(max_features=10000), "meta_description")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])
pipe.fit(data_train_exp, y_train)
y_pred = pipe.predict(data_val_exp)
print(f1_score(y_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.947866867003597
0.8398407689293155


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("title", CountVectorizer(max_features=10000), "title"),
        ("meta_keywords", CountVectorizer(max_features=10000), "meta_keywords")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])
pipe.fit(data_train_exp, y_train)
y_pred = pipe.predict(data_val_exp)
print(f1_score(y_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.9548310987091359
0.8379861026453737


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("title", CountVectorizer(max_features=10000), "title")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])
pipe.fit(data_train_exp, y_train)
y_pred = pipe.predict(data_val_exp)
print(f1_score(y_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.9440532524282835
0.8088066722004822


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("meta_keywords", CountVectorizer(max_features=10000), "meta_keywords")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])
pipe.fit(data_train_exp, y_train)
y_pred = pipe.predict(data_val_exp)
print(f1_score(y_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.9400105284841271
0.8339243131156039


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(max_features=10000), "content_tokens_stem"),
        ("meta_description", CountVectorizer(max_features=10000), "meta_description")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])
pipe.fit(data_train_exp, y_train)
y_pred = pipe.predict(data_val_exp)
print(f1_score(y_val, y_pred=y_pred))
y_pred = pipe.predict(data_test_exp)
print(f1_score(y_test, y_pred=y_pred))

0.9376565417368733
0.8193217440866344


In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("combined", CountVectorizer(max_features=10000), "combined")
    ]
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", Log_model)
])

In [17]:
X_train = X_train.copy()
X_val = X_val.copy()
df_test = df_test.copy()

X_train["combined"] = (
    X_train["content_tokens_stem"].fillna("").astype(str) + " " +
    X_train["title"].fillna("").astype(str) + " " +
    X_train["meta_description"].fillna("").astype(str) + " " +
    X_train["meta_keywords"].fillna("").astype(str)
).fillna("")

X_val["combined"] = (
    X_val["content_tokens_stem"].fillna("").astype(str) + " " +
    X_val["title"].fillna("").astype(str) + " " +
    X_val["meta_description"].fillna("").astype(str) + " " +
    X_val["meta_keywords"].fillna("").astype(str)
).fillna("")

df_test["combined"] = (
    df_test["content_tokens_stem"].fillna("").astype(str) + " " +
    df_test["title"].fillna("").astype(str) + " " +
    df_test["meta_description"].fillna("").astype(str) + " " +
    df_test["meta_keywords"].fillna("").astype(str)
).fillna("")

In [18]:
data_train_exp_2 = X_train[["combined"]].copy()
data_val_exp_2 = X_val[["combined"]].copy()
data_test_exp_2 = df_test[["combined"]].copy()
pipe.fit(data_train_exp_2, y_train)
y_pred = pipe.predict(data_val_exp_2)
print(f1_score(y_val, y_pred))
y_pred = pipe.predict(data_test_exp_2)
print(f1_score(y_test, y_pred))

0.9452648293070205
0.8232230454004078
